In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *


In [0]:
dbutils.widgets.text('incremental_flag','0')

In [0]:
incremental_flag=dbutils.widgets.get('incremental_flag')

In [0]:
df_src=spark.sql("""select DISTINCT(Dealer_ID) AS dealer_id, DealerName as dealer_name from parquet.`abfss://silver@carstorage1234.dfs.core.windows.net/rawdata`""")
df_src.display()

dealer_id,dealer_name
DLR0069,Geo Motors
DLR0249,Acura Motors
DLR0209,Zastava Motors
DLR0189,Sunbeam Motors
DLR0130,Micro Motors
DLR0093,Iso Motors
DLR0135,Morgan Motors
DLR0170,Saleen Motors
DLR0265,Bentley Motors
DLR0087,Hyundai Motors


In [0]:
if spark.catalog.tableExists('prod_car.gold.dim_dealer'):
    df_sink = spark.sql('''select  
    dim_dealer_key, dealer_id, dealer_name from prod_car.gold.dim_dealer''')
else:    
    df_sink = spark.sql('''select 1 as 
    dim_dealer_key,Dealer_ID as dealer_id,DealerName as Dealer_name from parquet.`abfss://silver@carstorage1234.dfs.core.windows.net/rawdata` where 1=0''')

In [0]:
df_sink.display()

dim_dealer_key,dealer_id,Dealer_name


In [0]:
df_filter=df_src.join(df_sink,df_src['dealer_id']==df_sink['dealer_id'], 'left').select(df_src['dealer_id'],df_src['dealer_name'],df_sink['dim_dealer_key'])
df_filter.display()

dealer_id,dealer_name,dim_dealer_key
DLR0069,Geo Motors,null
DLR0249,Acura Motors,null
DLR0209,Zastava Motors,null
DLR0189,Sunbeam Motors,null
DLR0130,Micro Motors,null
DLR0093,Iso Motors,null
DLR0135,Morgan Motors,null
DLR0170,Saleen Motors,null
DLR0265,Bentley Motors,null
DLR0087,Hyundai Motors,null


In [0]:
df_filter_old=df_filter.filter(col("dim_dealer_key").isNotNull())
df_filter_old.display()

dealer_id,dealer_name,dim_dealer_key


In [0]:
df_filter_new=df_filter.filter(col("dim_dealer_key").isNull())
df_filter_new.display()

dealer_id,dealer_name,dim_dealer_key
DLR0069,Geo Motors,null
DLR0249,Acura Motors,null
DLR0209,Zastava Motors,null
DLR0189,Sunbeam Motors,null
DLR0130,Micro Motors,null
DLR0093,Iso Motors,null
DLR0135,Morgan Motors,null
DLR0170,Saleen Motors,null
DLR0265,Bentley Motors,null
DLR0087,Hyundai Motors,null


In [0]:
if (incremental_flag=='0'):
    max_value = 1
else: 
    max_value_df=spark.sql("select max(dim_dealer_key) from prod_car.gold.dim_dealer ")
    max_value=max_value_df.collect()[0][0]+1


In [0]:
df_filter_new=df_filter_new.withColumn("dim_dealer_key",max_value+monotonically_increasing_id()+1)
df_filter_new.display()

dealer_id,dealer_name,dim_dealer_key
DLR0069,Geo Motors,2
DLR0249,Acura Motors,3
DLR0209,Zastava Motors,4
DLR0189,Sunbeam Motors,5
DLR0130,Micro Motors,6
DLR0093,Iso Motors,7
DLR0135,Morgan Motors,8
DLR0170,Saleen Motors,9
DLR0265,Bentley Motors,10
DLR0087,Hyundai Motors,11


In [0]:
df_final=df_filter_new.union(df_filter_old)

In [0]:
if spark.catalog.tableExists('prod_car.gold.dim_dealer'):
    delta_tbl=DeltaTable.forPath(spark,"abfss://gold@carstorage1234.dfs.core.windows.net/dim_dealer")
    delta_tbl.alias("t").merge(df_final.alias("s"),"t.dim_dealer_key=s.dim_dealer_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta")\
            .mode('overwrite')\
            .option("path","abfss://gold@carstorage1234.dfs.core.windows.net/dim_dealer")\
            .saveAsTable("prod_car.gold.dim_dealer")